# Pegasus Fine-Tuning: Qwen 2.5 3B -> Hermes Tool-Calling Expert

This notebook fine-tunes Qwen 2.5 3B on the Hermes function-calling dataset using Unsloth.

**Requirements:** Google Colab with T4 GPU (free tier works)

**Output:** A GGUF file you can drop into Pegasus on your iPhone.

In [ ]:
# Cell 1: Install dependencies
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
# Cell 2: Run the fine-tuning script
!python /content/finetune_qwen_hermes.py

**OR** run it inline below (same code, cell by cell):

In [ ]:
# Cell 3: Load model
from unsloth import FastLanguageModel
import json

MAX_SEQ_LENGTH = 4096
LORA_RANK = 64
OUTPUT_DIR = "/content/pegasus_qwen_finetuned"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Model loaded!")

In [ ]:
# Cell 4: Load and format dataset
from datasets import load_dataset

PEGASUS_TOOLS = [
    {"type": "function", "function": {"name": "web_search", "description": "Search the web", "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
    {"type": "function", "function": {"name": "web_fetch", "description": "Fetch URL content", "parameters": {"type": "object", "properties": {"url": {"type": "string"}}, "required": ["url"]}}},
    {"type": "function", "function": {"name": "file_read", "description": "Read a file", "parameters": {"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"]}}},
    {"type": "function", "function": {"name": "file_write", "description": "Write a file", "parameters": {"type": "object", "properties": {"path": {"type": "string"}, "content": {"type": "string"}}, "required": ["path", "content"]}}},
    {"type": "function", "function": {"name": "python_exec", "description": "Run Python code", "parameters": {"type": "object", "properties": {"code": {"type": "string"}}, "required": ["code"]}}},
    {"type": "function", "function": {"name": "memory_read", "description": "Read persistent memory", "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {"name": "memory_write", "description": "Write to memory", "parameters": {"type": "object", "properties": {"content": {"type": "string"}}, "required": ["content"]}}},
    {"type": "function", "function": {"name": "shell_exec", "description": "Run shell command", "parameters": {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]}}},
]
TOOLS_STR = json.dumps(PEGASUS_TOOLS, indent=2)

dataset = load_dataset("NousResearch/hermes-function-calling-v1", split="train")

def format_hermes_conversation(example):
    conversations = example.get("conversations", [])
    if not conversations:
        return {"text": ""}
    parts = []
    has_tool_call = False
    for msg in conversations:
        role = msg.get("from", msg.get("role", ""))
        content = msg.get("value", msg.get("content", ""))
        if not content:
            continue
        if role == "system":
            sc = content
            if "function" in content.lower() or "tool" in content.lower():
                sc = f"{content}\n\nYou have access to the following tools:\n<tools>\n{TOOLS_STR}\n</tools>\n\nTo call a tool, output a JSON object inside <tool_call></tool_call> tags.\nDo NOT narrate or explain which tool you will use. Just call it directly."
            parts.append(f"<|im_start|>system\n{sc}<|im_end|>")
        elif role == "human":
            parts.append(f"<|im_start|>user\n{content}<|im_end|>")
        elif role == "gpt":
            formatted = content
            if '"name"' in content and '"arguments"' in content:
                has_tool_call = True
                if "<tool_call>" not in content:
                    try:
                        parsed = json.loads(content)
                        if isinstance(parsed, dict) and "name" in parsed:
                            formatted = f"<tool_call>\n{json.dumps(parsed)}\n</tool_call>"
                        elif isinstance(parsed, list):
                            formatted = "\n".join(f"<tool_call>\n{json.dumps(tc)}\n</tool_call>" for tc in parsed if isinstance(tc, dict) and "name" in tc)
                    except json.JSONDecodeError:
                        pass
            parts.append(f"<|im_start|>assistant\n{formatted}<|im_end|>")
        elif role == "tool":
            parts.append(f"<|im_start|>tool\n{content}<|im_end|>")
    text = "\n".join(parts)
    if not has_tool_call and "tool_call" not in text.lower():
        return {"text": ""}
    return {"text": text}

formatted_dataset = dataset.map(format_hermes_conversation, num_proc=2)
formatted_dataset = formatted_dataset.filter(lambda x: len(x["text"]) > 100)
print(f"Training examples: {len(formatted_dataset)}")
print("\nSample:")
print(formatted_dataset[0]["text"][:800])

In [ ]:
# Cell 5: Train
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=50,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=25,
        save_strategy="steps",
        save_steps=500,
        optim="adamw_8bit",
        seed=42,
        report_to="none",
    ),
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=True,
)

trainer.train()
print("Training complete!")

In [ ]:
# Cell 6: Export to GGUF
print("Saving model...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Exporting to GGUF (q8_0)...")
model.save_pretrained_gguf(
    OUTPUT_DIR,
    tokenizer,
    quantization_method="q8_0",
)
print("Done! GGUF file is in:", OUTPUT_DIR)

In [ ]:
# Cell 7: Download the GGUF file
import glob
gguf_files = glob.glob(f"{OUTPUT_DIR}/*.gguf")
print("GGUF files:", gguf_files)

# Download from Colab
if gguf_files:
    from google.colab import files
    files.download(gguf_files[0])
    print(f"Downloading {gguf_files[0]}...")